# MSSAD-style testing on MedIAnomaly: Grubbs and Rosner (no calibration set)

**Goal.** Reproduce the earlier MedIAnomaly results of the paper (UAI/ICDM version), where reconstruction errors of AE-d* are tested with Grubbs' or Rosner's test against the **training errors** (no calibration set), and compare raw errors with Box-Cox-transformed errors.

**Training.** AE-d* (official MedIAnomaly code, d* = 4, 4, 32, 16) on **all** training images, λ ∈ {0, 1e-4, 1e-3, 1e-2, 1e-1}, seeds 0–2 (60 runs). Training and test errors are saved, so every variant below is computed from the same models.

**Scoring variants (as described in the earlier paper):**
- **Grubbs:** each test error is appended to the training errors; score G = |e − mean| / sd of the augmented sample (two-sided, as in the paper; one-sided also reported); detection when G exceeds Grubbs' critical value for N+1 points.
- **Rosner (generalized ESD):** applied to the pooled training and test errors with up to r = `ROSNER_R_FRAC` × (number of test samples) outliers and Rosner's Bonferroni-type critical values; test samples among the detected outliers are flagged and scored by their ESD statistic, all others score 0. The earlier paper does not state the pooling or r, so both are explicit settings here.
- **Transform:** `raw` (as in the earlier paper) or `boxcox` (Box-Cox fitted on the training errors, applied to training and test errors).

**Reported per dataset (mean ± std over seeds):** AUROC/AP and F1 / precision / recall / FPR at α = 0.05 and 0.10, for every λ, for the union over λ, and at the **best λ selected on the test set**, which is how the earlier paper reported its tables. The FPR on the normal test images is reported for every variant, so the operating point of each result is visible.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
!pip install -q SimpleITK
import os, sys, time, json, random, glob, subprocess
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

if not os.path.exists('/content/MedIAnomaly'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/caiyu6666/MedIAnomaly.git', '/content/MedIAnomaly'], check=True)
sys.path.insert(0, '/content/MedIAnomaly/reconstruction')
from networks.ae import AE                          # official architecture
from dataloaders.dataload import MedAD, BraTSAD     # official data loaders and splits

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

In [ ]:
# ── Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/STRAP_MedIAnomaly'
RUNS_DIR   = os.path.join(DRIVE_ROOT, 'runs_full_all')  # AE-d* on all training images, every lambda
TAR_CACHE  = os.path.join(DRIVE_ROOT, 'data_tar')       # cached Zenodo archives (shared with earlier runs)
os.makedirs(RUNS_DIR, exist_ok=True); os.makedirs(TAR_CACHE, exist_ok=True)
print('Outputs:', DRIVE_ROOT)

In [ ]:
# ── Data: MedIAnomaly pre-processed datasets (Zenodo record 12677223) ───
DATA_ROOT = os.path.expanduser('~/MedIAnomaly-Data')     # path expected by the official loaders
ARCHIVES  = {'rsna': ('RSNA.tar.gz', 'RSNA'), 'vin': ('VinCXR.tar.gz', 'VinCXR'),
             'brain': ('BrainTumor.tar.gz', 'BrainTumor'), 'brats': ('BraTS2021.tar.gz', 'BraTS2021')}
ZENODO = 'https://zenodo.org/records/12677223/files/{}?download=1'
os.makedirs(DATA_ROOT, exist_ok=True)

for ds, (tar, folder) in ARCHIVES.items():
    if os.path.isdir(os.path.join(DATA_ROOT, folder)):
        print(f'{ds}: ready'); continue
    cached = os.path.join(TAR_CACHE, tar)
    if not os.path.exists(cached):
        print(f'{ds}: downloading {tar} ...')
        subprocess.run(['wget', '-q', '-O', cached, ZENODO.format(tar)], check=True)
    print(f'{ds}: extracting ...')
    subprocess.run(['tar', '-xzf', cached, '-C', DATA_ROOT], check=True)
    assert os.path.isdir(os.path.join(DATA_ROOT, folder)), f'unexpected archive layout for {tar}'
print(os.listdir(DATA_ROOT))

In [ ]:
# ── Configuration (official MedIAnomaly settings) ──────────────────────
DATASETS   = ['rsna', 'vin', 'brain', 'brats']
D_STAR     = {'rsna': 4, 'vin': 4, 'brain': 32, 'brats': 16}      # Cai et al. (2024), AE-d*
LAMBDAS    = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
SEEDS      = [0, 1, 2]
EPOCHS     = {'rsna': 250, 'vin': 250, 'brain': 600, 'brats': 250}
BATCH, LR, INPUT_SIZE, HIDDEN = 64, 1e-3, 64, 1024
ALPHAS     = [0.05, 0.10]
ROSNER_R_FRAC = 0.5                                       # Rosner: at most this share of the test set as outliers

def lam_tag(l): return 'lam0' if l == 0 else f'lam1e{abs(int(round(np.log10(l))))}'
def run_file(ds, seed, lam): return os.path.join(RUNS_DIR, f'{ds}_d{D_STAR[ds]}_seed{seed}_{lam_tag(lam)}.npz')
print(len(DATASETS) * len(LAMBDAS) * len(SEEDS), 'runs')

In [ ]:
# ── Data loading, training and reconstruction errors ───────────────────
TRANSFORM = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
_DATA = {}
def get_sets(ds):
    if ds not in _DATA:
        path = os.path.join(DATA_ROOT, ARCHIVES[ds][1])
        cls = BraTSAD if ds == 'brats' else MedAD
        _DATA[ds] = (cls(main_path=path, img_size=INPUT_SIZE, transform=TRANSFORM, mode='train'),
                     cls(main_path=path, img_size=INPUT_SIZE, transform=TRANSFORM, mode='test'))
    return _DATA[ds]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def build_ae(d):
    return AE(input_size=INPUT_SIZE, in_planes=1, base_width=16, expansion=1, mid_num=HIDDEN,
              latent_size=d, en_num_layers=1, de_num_layers=1).to(DEVICE)

def train(net, train_set, lam, epochs):
    # drop a final batch of size 1, which BatchNorm1d cannot train on
    loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,
                        drop_last=(len(train_set) % BATCH == 1))
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=0)
    net.train()
    for _ in range(epochs):
        for batch in loader:
            x = batch['img'].to(DEVICE)
            out = net(x)
            loss = ((x - out['x_hat']) ** 2).mean() + lam * out['z'].abs().mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return net

@torch.no_grad()
def image_errors(net, dataset):
    net.eval(); errs, labels = [], []
    for batch in DataLoader(dataset, batch_size=256, shuffle=False):
        x = batch['img'].to(DEVICE)
        errs.append(((x - net(x)['x_hat']) ** 2).mean(dim=[1, 2, 3]).cpu().numpy())
        labels.append(np.asarray(batch['label']))
    return np.concatenate(errs), np.concatenate(labels)
print('Helpers ready.')

In [ ]:
# ── Main loop (resumable): AE-d* on ALL training images ────────────────
todo = [(ds, s, l) for s in SEEDS for ds in DATASETS for l in LAMBDAS if not os.path.exists(run_file(ds, s, l))]
print(f'{len(todo)} runs to do')
for i, (ds, seed, lam) in enumerate(todo):
    t0 = time.time()
    train_set, test_set = get_sets(ds)
    set_seed(seed)
    net = train(build_ae(D_STAR[ds]), train_set, lam, EPOCHS[ds])
    tr_err, _ = image_errors(net, train_set)
    te_err, y = image_errors(net, test_set)
    np.savez_compressed(run_file(ds, seed, lam), train_err=tr_err.astype(np.float32),
                        test_err=te_err.astype(np.float32), y=y.astype(np.int8))
    print(f'[{i+1}/{len(todo)}] {ds} seed={seed} {lam_tag(lam)}: raw-error AUROC={roc_auc_score(y, te_err):.4f} ({time.time()-t0:.0f}s)')
    del net; torch.cuda.empty_cache()
print('Done.')

In [ ]:
# ── Analysis: Grubbs and Rosner, raw and Box-Cox ───────────────────────
from scipy.special import boxcox as boxcox_transform

def grubbs_crit(n, alpha, two_sided=True):
    t = stats.t.ppf(1 - alpha / (2 * n) if two_sided else 1 - alpha / n, df=n - 2)
    return ((n - 1) / np.sqrt(n)) * np.sqrt(t**2 / (n - 2 + t**2))

def grubbs_scores(tr, te, two_sided=True):
    # each test error appended to the training errors (augmented sample of size N+1)
    N = len(tr); S1, S2 = tr.sum(), (tr**2).sum()
    mu = (S1 + te) / (N + 1); sd = np.sqrt(np.maximum((S2 + te**2 - (N + 1) * mu**2) / N, 0))
    dev = np.abs(te - mu) if two_sided else (te - mu)
    return np.where(sd > 1e-12, dev / sd, 0.0)

def rosner(tr, te, alpha, r_frac=ROSNER_R_FRAC):
    # generalized ESD on the pooled training and test errors (two-sided)
    x = np.r_[tr, te]; is_test = np.r_[np.zeros(len(tr), bool), np.ones(len(te), bool)]
    n = len(x); r = max(1, int(np.ceil(r_frac * len(te))))
    alive = np.ones(n, bool); S1, S2, m = x.sum(), (x**2).sum(), n
    removed, R, lam_crit = [], [], []
    for i in range(1, r + 1):
        mu = S1 / m; sd = np.sqrt(max((S2 - m * mu**2) / (m - 1), 1e-300))
        dev = np.where(alive, np.abs(x - mu), -np.inf); j = int(np.argmax(dev))
        R.append(dev[j] / sd); removed.append(j)
        p = 1 - alpha / (2 * (n - i + 1)); t = stats.t.ppf(p, n - i - 1)
        lam_crit.append((n - i) * t / np.sqrt((n - i - 1 + t**2) * (n - i + 1)))
        alive[j] = False; S1 -= x[j]; S2 -= x[j]**2; m -= 1
    R, lam_crit = np.array(R), np.array(lam_crit)
    n_out = int(np.max(np.where(R > lam_crit)[0]) + 1) if (R > lam_crit).any() else 0
    score = np.zeros(len(te)); det = np.zeros(len(te), bool)
    for k in range(n_out):
        j = removed[k]
        if is_test[j]: score[j - len(tr)] = R[k]; det[j - len(tr)] = True
    return score, det

def prf(y, det):
    tp = int((det & (y == 1)).sum()); fp = int((det & (y == 0)).sum())
    p = tp / max(1, int(det.sum())); r = tp / max(1, int(y.sum()))
    return {'f1': 0.0 if p + r == 0 else 2 * p * r / (p + r), 'precision': p, 'recall': r, 'fpr': fp / max(1, int((y == 0).sum()))}

def transform(tr, te, kind):
    if kind == 'raw': return tr, te
    trb, lmb = stats.boxcox(np.maximum(tr, 1e-12)); return trb, boxcox_transform(np.maximum(te, 1e-12), lmb)

rows = []
for ds in DATASETS:
    for seed in SEEDS:
        files = [run_file(ds, seed, l) for l in LAMBDAS]
        if not all(os.path.exists(f) for f in files): continue
        R = [np.load(f) for f in files]; y = R[0]['y'].astype(int)
        for kind in ['raw', 'boxcox']:
            dets = {}
            for k, lam in enumerate(LAMBDAS):
                tr, te = transform(R[k]['train_err'].astype(np.float64), R[k]['test_err'].astype(np.float64), kind)
                for method, two in [('Grubbs (two-sided)', True), ('Grubbs (one-sided)', False)]:
                    sc = grubbs_scores(tr, te, two)
                    for a in ALPHAS:
                        d = sc > grubbs_crit(len(tr) + 1, a, two); dets.setdefault((method, a), []).append(d)
                        rows.append({'dataset': ds, 'seed': seed, 'transform': kind, 'method': method, 'lambda': lam_tag(lam), 'alpha': a,
                                     'auroc': roc_auc_score(y, sc), 'ap': average_precision_score(y, sc), **prf(y, d)})
                for a in ALPHAS:
                    sc, d = rosner(tr, te, a); dets.setdefault(('Rosner', a), []).append(d)
                    rows.append({'dataset': ds, 'seed': seed, 'transform': kind, 'method': 'Rosner', 'lambda': lam_tag(lam), 'alpha': a,
                                 'auroc': roc_auc_score(y, sc), 'ap': average_precision_score(y, sc), **prf(y, d)})
            for (method, a), D in dets.items():          # union over lambda (aggregated detections)
                rows.append({'dataset': ds, 'seed': seed, 'transform': kind, 'method': method, 'lambda': 'union', 'alpha': a,
                             'auroc': np.nan, 'ap': np.nan, **prf(y, np.any(D, axis=0))})
        print(f'{ds} seed {seed} done')

res = pd.DataFrame(rows); res.to_csv(os.path.join(DRIVE_ROOT, 'mssad_grubbs_rosner_per_seed.csv'), index=False)
g = res.groupby(['dataset', 'transform', 'method', 'lambda', 'alpha']).agg(['mean', 'std'])
g.columns = [f'{a}_{b}' for a, b in g.columns]; g = g.drop(columns=[c for c in g.columns if c.startswith('seed')]).reset_index()
g.to_csv(os.path.join(DRIVE_ROOT, 'mssad_grubbs_rosner_summary.csv'), index=False)

# best lambda per dataset, selected on the test set (as reported in the earlier paper)
pd.set_option('display.width', 250)
sel = g[(g['lambda'] != 'union') & (g.alpha == 0.05)]
best = sel.loc[sel.groupby(['dataset', 'transform', 'method']).auroc_mean.idxmax()]
print('AUROC / AP at the best lambda per dataset (selected on the TEST set, as in the earlier paper), alpha = 0.05')
print((best.set_index(['transform', 'method', 'dataset'])[['lambda', 'auroc_mean', 'auroc_std', 'ap_mean', 'f1_mean', 'fpr_mean']]
       .assign(**{c: lambda d, c=c: (100 * d[c]).round(1) for c in ['auroc_mean', 'auroc_std', 'ap_mean', 'f1_mean', 'fpr_mean']})).to_string())
print()
print('F1 / FPR (%) of the union over lambda, alpha = 0.05')
u = g[(g['lambda'] == 'union') & (g.alpha == 0.05)].set_index(['transform', 'method', 'dataset'])[['f1_mean', 'fpr_mean', 'recall_mean']]
print((100 * u).round(1).to_string())
print(); print('Saved: mssad_grubbs_rosner_per_seed.csv, mssad_grubbs_rosner_summary.csv')